# Модель

In [7]:
import pandas as pd
import numpy as np

# Загрузим данные
df_scoring = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\ML\scoring_results.csv')
df_train_scores = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\ML\train_scores.csv')

In [11]:
df_scoring.head()

,customer_id,snapshot_date,score,target_paid_30d,recovered_amount_30d,dpd_bucket,region,current_contact_flg,income_30d
0,200034,2026-02-01,0.184798,0,0.0,31-60,Ural,1,0.0
1,200182,2026-02-01,0.038820,0,0.0,31-60,Siberia,0,0.0
2,200215,2026-02-01,0.070687,0,0.0,90+,South,1,0.0
3,200225,2026-02-01,0.061727,0,0.0,90+,SPB,1,0.0
4,200243,2026-02-01,0.016273,0,0.0,90+,SPB,1,0.0


In [13]:
df_train_scores.head()

,customer_id,snapshot_date,score
0,100350,2026-01-01,0.223736
1,100479,2026-01-01,0.159427
2,100509,2026-01-01,0.282220
3,100562,2026-01-01,0.732975
4,100660,2026-01-01,0.895193


In [9]:
# Подготовим данные
df_scoring['snapshot_date'] = pd.to_datetime(df_scoring['snapshot_date'])
df_train_scores['snapshot_date'] = pd.to_datetime(df_train_scores['snapshot_date'])

# Расчитаем доход по клиенту:
df_scoring['income_30d'] = np.where(df_scoring['target_paid_30d'] == 1, df_scoring['recovered_amount_30d'], 0)

In [15]:
# Определим порог для топ-25% клиентов,с наибольшей вероятностью платежа 
threshold_25 = df_scoring['score'].quantile(0.75)

df_scoring['new_contact_flg'] = (df_scoring['score'] >= threshold_25).astype(int)

new_contact_share = df_scoring['new_contact_flg'].mean()

In [17]:
# Оценим экономический эффект от новой стратегии
df_new_strategy = df_scoring[df_scoring['new_contact_flg'] == 1].copy()

new_contacts_cnt = len(df_new_strategy)
new_recovered_sum = df_new_strategy['income_30d'].sum()
new_contact_cost = 35 * new_contacts_cnt
new_economic_effect = new_recovered_sum - new_contact_cost

In [19]:
# Расчитаем экономический эффект текущей стратегии
current_strategy_df = df_scoring[df_scoring['current_contact_flg'] == 1].copy()

current_contacts_cnt = len(current_strategy_df)
current_recovered_sum = current_strategy_df['income_30d'].sum()
current_contact_cost = 35 * current_contacts_cnt
current_economic_effect = current_recovered_sum - current_contact_cost
effect_delta = new_economic_effect - current_economic_effect

In [39]:
# Сравним обе стратегии
strategy_comparison = pd.DataFrame({'strategy': ['current_strategy', 'new_top25_strategy'], 'contacts_cnt': [current_contacts_cnt, new_contacts_cnt],
                                    'recovered_sum': [current_recovered_sum, new_recovered_sum], 'economic_effect': [current_economic_effect, new_economic_effect]})

print('\nСравнение стратегий')
print(strategy_comparison.round(2))
print('\n')
print(f'Порог для топ-25% клиентов: {threshold_25:.6f}')
print(f'Ожидаемый экономический эффект от новой стратегии: {new_economic_effect:.6f}')
print(f'Разница в экономическом эффекте между новой и старой стратегией: {effect_delta:.2f}')


Сравнение стратегий
             strategy  contacts_cnt  recovered_sum  economic_effect
0    current_strategy           768     4505520.09       4478640.09
1  new_top25_strategy           600     6265682.52       6244682.52


Порог для топ-25% клиентов: 0.429786
Ожидаемый экономический эффект от новой стратегии: 6244682.520000
Разница в экономическом эффекте между новой и старой стратегией: 1766042.43


In [47]:
# Сравним распределение score
bins = np.linspace(0, 1, 11)

train_bins = pd.cut(df_train_scores['score'], bins=bins, include_lowest=True)
scoring_bins = pd.cut(df_scoring['score'], bins=bins, include_lowest=True)

train_dist = train_bins.value_counts(normalize=True, sort=False)
scoring_dist = scoring_bins.value_counts(normalize=True, sort=False)

score_distribution_comparison = pd.DataFrame({'score_bin': train_dist.index.astype(str), 'train_share': train_dist.values, 
                                              'scoring_share': scoring_dist.values})

score_distribution_comparison['delta'] = (score_distribution_comparison['scoring_share'] -score_distribution_comparison['train_share'])
score_distribution_comparison['abs_delta'] = score_distribution_comparison['delta'].abs()

print('\nСравнение распределения score по бинам')
print(score_distribution_comparison.round(4))

largest_shifts = score_distribution_comparison.sort_values(by='abs_delta', ascending=False).head(3)

print('\nТоп-3 наибольших изменений в распределении score')
print(largest_shifts.round(4))



Сравнение распределения score по бинам
       score_bin  train_share  scoring_share   delta  abs_delta
0  (-0.001, 0.1]       0.1063         0.2342  0.1279     0.1279
1     (0.1, 0.2]       0.1643         0.2146  0.0503     0.0503
2     (0.2, 0.3]       0.1551         0.1571  0.0020     0.0020
3     (0.3, 0.4]       0.1459         0.1150 -0.0309     0.0309
4     (0.4, 0.5]       0.1172         0.0858 -0.0313     0.0313
5     (0.5, 0.6]       0.0959         0.0571 -0.0388     0.0388
6     (0.6, 0.7]       0.0735         0.0542 -0.0194     0.0194
7     (0.7, 0.8]       0.0597         0.0412 -0.0185     0.0185
8     (0.8, 0.9]       0.0460         0.0292 -0.0168     0.0168
9     (0.9, 1.0]       0.0362         0.0117 -0.0245     0.0245

Топ-3 наибольших изменений в распределении score
       score_bin  train_share  scoring_share   delta  abs_delta
0  (-0.001, 0.1]       0.1063         0.2342  0.1279     0.1279
1     (0.1, 0.2]       0.1643         0.2146  0.0503     0.0503
5     (0.5, 0.

In [65]:
# Посчитаем PSI
def calc_psi(expected, actual, bins=10):
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    quantile_points = np.linspace(0, 1, bins + 1)
    breakpoints = expected.quantile(quantile_points).values
    breakpoints = np.unique(breakpoints)

    if len(breakpoints) < 3:
        breakpoints = np.linspace(expected.min(), expected.max(), bins + 1)
        breakpoints = np.unique(breakpoints)

    expected_bins = pd.cut(expected, bins=breakpoints, include_lowest=True)
    actual_bins = pd.cut(actual, bins=breakpoints, include_lowest=True)

    expected_dist = expected_bins.value_counts(normalize=True, sort=False)
    actual_dist = actual_bins.value_counts(normalize=True, sort=False)

    psi_df = pd.DataFrame({'expected_pct': expected_dist, 'actual_pct': actual_dist}).fillna(0)
    psi_df['expected_pct'] = np.where(psi_df['expected_pct'] == 0, 0.0001, psi_df['expected_pct'])
    psi_df['actual_pct'] = np.where(psi_df['actual_pct'] == 0, 0.0001, psi_df['actual_pct'])
    psi_df['psi'] = ((psi_df['actual_pct'] - psi_df['expected_pct']) * np.log(psi_df['actual_pct'] / psi_df['expected_pct']))

    total_psi = psi_df['psi'].sum()
    psi_df = psi_df.reset_index().rename(columns={'index': 'bin'})

    return total_psi, psi_df

psi_value, psi_details = calc_psi(expected=df_train_scores['score'], actual=df_scoring['score'], bins=10)

print(f'\nPSI: {psi_value:.6f}')
print('\nДетализация PSI по бинам')
print(psi_details.round(6))


PSI: 0.198218

Детализация PSI по бинам
               score  expected_pct  actual_pct       psi
0  (0.00341, 0.0966]      0.100517    0.222593  0.097052
1    (0.0966, 0.159]      0.099943    0.146311  0.017673
2     (0.159, 0.216]      0.099943    0.114214  0.001905
3     (0.216, 0.284]      0.099943    0.102126  0.000047
4     (0.284, 0.344]      0.099943    0.071697  0.009382
5     (0.344, 0.428]      0.099943    0.091288  0.000784
6     (0.428, 0.508]      0.099943    0.065861  0.014214
7     (0.508, 0.624]      0.099943    0.060025  0.020351
8     (0.624, 0.763]      0.099943    0.070863  0.009999
9     (0.763, 0.983]      0.099943    0.055023  0.026810


Как мы видим по значению PSI - есть умеренный сдвиг распределения score.

Это означает, что профиль клиентов в текущих данных несколько отличается от обучающей выборки.

При таком уровне PSI модель остаётся работоспособной, однако её качество может постепенно ухудшаться, особенно на отдельных сегментах клиентов.

# Вывод

Новая стратегия обзвона топ-25% клиентов показывает существенный рост экономического эффекта: прибыль увеличивается примерно на 1.77 млн рублей по сравнению с текущей стратегией при меньшем количестве контактов. 

При этом наблюдается умеренный сдвиг распределения скоринга (PSI = 0.20), что указывает на изменение профиля клиентов — в частности, увеличилась доля клиентов с низкими score и снизилась доля высоких score. 

Несмотря на это, модель остаётся работоспособной и эффективно выделяет наиболее платёжеспособных клиентов. 
Рекомендуется запуск новой стратегии с обязательным мониторингом PSI и ключевых метрик качества для своевременного выявления возможной деградации.